# CropDetector — Phase 9, Stage B (Part 1)
## Supported-crop recognition with an explicit **Other Crop** class

**Standalone Google Colab notebook.** Upload only `CropDetector_Phase9_Part2_Data.zip` from your previous run. No retraining of Stage A or the 15-class disease classifier is needed. **GPU recommended**.

**Output:** `CropDetector_Phase9_StageB_Part1_Results.zip` with a `.keras` checkpoint, per-image scores, development metrics, a separate never-trained-on-species diagnostic, and provenance reports.

Five labels in fixed order: **Maize, Grape, Potato, Tomato, Other Crop**. Unsupported tomato conditions are labelled **Tomato**, never **Other Crop**. We pre-register **Peach, Raspberry and Strawberry** as entirely unseen crop species and exclude *all* their examples from model fitting and threshold selection.

**Important:** This is a development-stage closed-set model with an incomplete `Other Crop` class, not a general-purpose unknown-species detector or field diagnostic guarantee. The held-out species results are evaluated once as a stress test; do not tune the model based on them.

## Cell 1 — Reproducible setup

In [ ]:
%pip -q install pillow scikit-learn

import os, io, json, sys, subprocess, hashlib, shutil, math
from pathlib import Path
from zipfile import ZipFile
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from scipy.special import softmax
from google.colab import files

SEED=42
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
CLASSES=['Maize','Grape','Potato','Tomato','Other Crop']
LABEL={name:i for i,name in enumerate(CLASSES)}
OTHER=LABEL['Other Crop']
UNSEEN_SPECIES=['Peach','Raspberry','Strawberry']
IMG_SIZE=(160,160)
BATCH_SIZE=32
ROOT=Path('/content')
OUT=ROOT/'CropDetector_Phase9_StageB_Part1'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'TensorFlow',tf.__version__)
print('GPU:',tf.config.list_physical_devices('GPU'))

## Cell 2 — Upload and validate Part 2 dataset manifests

In [ ]:
SOURCE_ZIP=ROOT/'CropDetector_Phase9_Part2_Data.zip'
if not SOURCE_ZIP.exists():
    uploaded=files.upload()
    assert SOURCE_ZIP.name in uploaded,'Upload CropDetector_Phase9_Part2_Data.zip'
with ZipFile(SOURCE_ZIP) as archive:
    needed={'audit_report.json','validator_train.csv','validator_dev.csv'}
    assert needed.issubset(archive.namelist()),'Required Part 2 files absent'
    audit=json.loads(archive.read('audit_report.json'))
    original_train=pd.read_csv(io.BytesIO(archive.read('validator_train.csv')))
    original_dev=pd.read_csv(io.BytesIO(archive.read('validator_dev.csv')))
assert audit['phase']=='Phase9_Part2'
assert len(original_train)==4065 and len(original_dev)==1012
assert set(original_train['Group']).isdisjoint(set(original_dev['Group']))
assert set(original_train['SHA256']).isdisjoint(set(original_dev['SHA256']))
assert {'PlantDoc','NaturalImages'}==set(original_train.Source)
assert original_train.loc[original_train.Source.eq('PlantDoc'),'Target'].eq(0).all()
assert original_dev.loc[original_dev.Source.eq('PlantDoc'),'Target'].eq(0).all()
print('Source PlantDoc commit:',audit['plantdoc_git_commit'])
print('Source ZIP SHA256:',hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest())

## Cell 3 — Build Stage B split; hold out entire crop species

**Why:** A random image split alone would put photographs of the same unsupported crop species in both training and development. The held-out group measures whether `Other Crop` works on **previously unseen species**. The labels of the held-out examples never influence model or threshold selection.

In [ ]:
train_all=original_train.loc[original_train.Source.eq('PlantDoc')].copy()
dev_all=original_dev.loc[original_dev.Source.eq('PlantDoc')].copy()

def unseen_mask(df):
    return df.Folder.str.lower().str.startswith(tuple(s.lower() for s in UNSEEN_SPECIES))

unseen_train=train_all.loc[unseen_mask(train_all)].copy()
unseen_dev=dev_all.loc[unseen_mask(dev_all)].copy()
unseen=pd.concat([unseen_train,unseen_dev],ignore_index=True)
train_df=train_all.loc[~unseen_mask(train_all)].copy().reset_index(drop=True)
dev_df=dev_all.loc[~unseen_mask(dev_all)].copy().reset_index(drop=True)
assert len(unseen)>0
assert unseen.CropLabel.eq('Other Crop').all()
assert train_df.CropLabel.isin(CLASSES).all() and dev_df.CropLabel.isin(CLASSES).all()
for frame in (train_df,dev_df,unseen):
    frame['CropTarget']=frame.CropLabel.map(LABEL).astype('int32')
assert set(train_df.Group).isdisjoint(set(dev_df.Group))
assert set(train_df.SHA256).isdisjoint(set(dev_df.SHA256))
assert set(unseen.Group).isdisjoint(set(train_df.Group)|set(dev_df.Group))
assert set(unseen.SHA256).isdisjoint(set(train_df.SHA256)|set(dev_df.SHA256))
assert not unseen_mask(train_df).any() and not unseen_mask(dev_df).any()
assert (train_df.loc[train_df.Role.eq('Unsupported Condition'),'CropLabel']=='Tomato').all()
assert (dev_df.loc[dev_df.Role.eq('Unsupported Condition'),'CropLabel']=='Tomato').all()

print('Training:',len(train_df),'Development:',len(dev_df),'Unseen crop species test:',len(unseen))
print('Held-out image counts by species:')
display(unseen.groupby('Folder').size().to_frame('images'))
summary=pd.DataFrame({'Train':train_df.CropLabel.value_counts(),
                      'Dev':dev_df.CropLabel.value_counts()}).reindex(CLASSES).fillna(0).astype(int)
display(summary)
summary.to_csv(OUT/'stageB_split_counts.csv')
train_df.to_csv(OUT/'stageB_train_manifest.csv',index=False)
dev_df.to_csv(OUT/'stageB_dev_manifest.csv',index=False)
unseen.to_csv(OUT/'stageB_unseen_species_manifest.csv',index=False)

## Cell 4 — Restore the exact PlantDoc image repository

All images and labels are reconstructed from your immutable Part 2 manifests, not the previous Colab session.

In [ ]:
COMMIT=audit['plantdoc_git_commit']
PLANTDOC_ROOT=ROOT/'PlantDoc-Dataset'
REPO='https://github.com/pratikkayal/PlantDoc-Dataset.git'
if not (PLANTDOC_ROOT/'.git').exists():
    subprocess.run(['git','clone','--depth','1',REPO,str(PLANTDOC_ROOT)],check=True)
head=subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
if head!=COMMIT:
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'fetch','--depth','1','origin',COMMIT],check=True)
    subprocess.run(['git','-C',str(PLANTDOC_ROOT),'checkout','--detach',COMMIT],check=True)
head=subprocess.check_output(['git','-C',str(PLANTDOC_ROOT),'rev-parse','HEAD'],text=True).strip()
assert head==COMMIT
print('Using pinned PlantDoc commit',head)

## Cell 5 — Verify exact image SHA-256 fingerprints

In [ ]:
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for part in iter(lambda:f.read(1024*1024),b''):
            h.update(part)
    return h.hexdigest()
for frame in (train_df,dev_df,unseen):
    frame['ResolvedPath']=frame.RelativePath.map(lambda rel:str(PLANTDOC_ROOT/str(rel)))
    assert all(Path(p).is_file() for p in frame.ResolvedPath),'PlantDoc files missing'
all_stageB=pd.concat([train_df,dev_df,unseen],ignore_index=True)
bad=[]
for i,r in enumerate(all_stageB.itertuples(index=False),1):
    if sha256_file(r.ResolvedPath).lower()!=str(r.SHA256).lower():
        bad.append(r.RelativePath)
    if i%500==0: print('Fingerprint checked',i,'/',len(all_stageB))
assert not bad,f'Image content changed; do not proceed: {bad[:4]}'
print('All',len(all_stageB),'PlantDoc images verified')

## Cell 6 — Prepare image datasets (pixel values 0–255, RGB 160×160)

In [ ]:
AUTOTUNE=tf.data.AUTOTUNE

def pil_decode_resize(path_bytes):
    p=os.fsdecode(path_bytes)
    with Image.open(p) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im=im.resize((160,160),resample=Image.Resampling.BILINEAR)
        return np.asarray(im,dtype=np.uint8)

def decode_image(path,y):
    x=tf.numpy_function(pil_decode_resize,[path],tf.uint8)
    x.set_shape([160,160,3])
    return tf.cast(x,tf.float32),tf.cast(y,tf.int32)

def augment_image(x,y):
    x=tf.image.random_flip_left_right(x)
    x=tf.image.random_brightness(x,max_delta=12.0)
    x=tf.image.random_contrast(x,lower=0.85,upper=1.15)
    return tf.clip_by_value(x,0.,255.),y

def make_ds(frame,training=False):
    ds=tf.data.Dataset.from_tensor_slices((frame.ResolvedPath.to_numpy(dtype=str),
                                           frame.CropTarget.to_numpy(dtype=np.int32)))
    if training: ds=ds.shuffle(len(frame),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(decode_image,num_parallel_calls=AUTOTUNE)
    if training: ds=ds.map(augment_image,num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds=make_ds(train_df,True)
dev_ds=make_ds(dev_df,False)
unseen_ds=make_ds(unseen,False)
N=len(train_df)
class_count=train_df.CropTarget.value_counts().to_dict()
class_weights={i:float(min(3.5,N/(5*class_count[i]))) for i in range(5)}
print('Class weights:',class_weights)
for x,y in train_ds.take(1):print('Input:',x.shape,'Targets:',y.shape,'RGB range:',float(tf.reduce_min(x)),float(tf.reduce_max(x)))

## Cell 7 — ImageNet MobileNetV2-0.35 five-class model

The entire model graph includes normalization; TFLite conversion will later accept **raw RGB 0–255** just like our existing Android Stage A.

In [ ]:
from tensorflow.keras import layers

backbone=tf.keras.applications.MobileNetV2(
    input_shape=(160,160,3),alpha=0.35,include_top=False,weights='imagenet')
backbone.trainable=False
image_input=tf.keras.Input(shape=(160,160,3),dtype=tf.float32,name='rgb_image')
x=layers.Rescaling(1./127.5,offset=-1.,name='mobilenet_preprocess')(image_input)
x=backbone(x,training=False)
x=layers.GlobalAveragePooling2D(name='global_pool')(x)
x=layers.Dropout(0.25,name='dropout')(x)
outputs=layers.Dense(5,name='crop_logits')(x)
model=tf.keras.Model(image_input,outputs,name='CropDetector_StageB_CropRecognition')

def compile_model(m,lr):
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
compile_model(model,1e-3)
model.summary()

## Cell 8 — Train classification head, with pretrained backbone frozen

In [ ]:
HEAD=OUT/'best_stageB_head.keras'
head_callbacks=[
  tf.keras.callbacks.ModelCheckpoint(str(HEAD),save_best_only=True,monitor='val_loss',mode='min'),
  tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=3,restore_best_weights=True),
  tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=2,min_lr=1e-5)]
head_history=model.fit(train_ds,validation_data=dev_ds,epochs=12,
                       class_weight=class_weights,callbacks=head_callbacks)
print('Frozen-head checkpoint:',HEAD)

## Cell 9 — Fine-tune only last 25 non-BN backbone layers

In [ ]:
ft=tf.keras.models.load_model(str(HEAD),compile=False)
ft_backbone=ft.get_layer(backbone.name)
ft_backbone.trainable=True
for layer in ft_backbone.layers: layer.trainable=False
for layer in ft_backbone.layers[-25:]:
    if not isinstance(layer,tf.keras.layers.BatchNormalization):
        layer.trainable=True
compile_model(ft,1e-5)
FT=OUT/'best_stageB_finetuned.keras'
ft_callbacks=[
  tf.keras.callbacks.ModelCheckpoint(str(FT),save_best_only=True,monitor='val_loss',mode='min'),
  tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=2,restore_best_weights=True)]
ft_history=ft.fit(train_ds,validation_data=dev_ds,epochs=6,
                  class_weight=class_weights,callbacks=ft_callbacks)
print('Fine-tuned checkpoint:',FT)

## Cell 10 — Select checkpoint using development loss only

`Unseen species` is *not* examined here to avoid leaking test feedback into model choice.

In [ ]:
def get_logits(checkpoint,ds):
    m=tf.keras.models.load_model(str(checkpoint),compile=False)
    return np.asarray(m.predict(ds.map(lambda x,y:x),verbose=0),dtype=np.float32)

ydev=dev_df.CropTarget.to_numpy(dtype=int)
rows=[]
logits_by_model={}
for name,path in [('Frozen head',HEAD),('Fine-tuned',FT)]:
    logits=get_logits(path,dev_ds)
    logits_by_model[name]=logits
    log_probs=logits-tf.reduce_logsumexp(logits,axis=1,keepdims=True).numpy()
    loss=float(-np.mean(log_probs[np.arange(len(ydev)),ydev]))
    pred=logits.argmax(axis=1)
    rows.append({'Checkpoint':name,'Development NLL':loss,
                 'Development accuracy':float(np.mean(pred==ydev)),
                 'Development balanced accuracy':float(balanced_accuracy_score(ydev,pred))})
comparison=pd.DataFrame(rows)
display(comparison)
comparison.to_csv(OUT/'checkpoint_comparison.csv',index=False)
chosen=comparison.sort_values('Development NLL').iloc[0]['Checkpoint']
BEST=HEAD if chosen=='Frozen head' else FT
BEST_MODEL=tf.keras.models.load_model(str(BEST),compile=False)
dev_logits=logits_by_model[chosen]
dev_prob=softmax(dev_logits.astype(np.float64),axis=1)
dev_pred=dev_prob.argmax(axis=1)
print('Selected checkpoint:',chosen)
print(classification_report(ydev,dev_pred,labels=list(range(5)),target_names=CLASSES,zero_division=0))
print('Confusion matrix: rows=actual, cols=predicted; order=',CLASSES)
display(pd.DataFrame(confusion_matrix(ydev,dev_pred,labels=list(range(5))),index=CLASSES,columns=CLASSES))

## Cell 11 — Set crop-gate threshold from development images **only**

A crop is allowed to enter the disease classifier only if **Stage B predicts one of the 4 supported crops AND its probability exceeds this threshold**. For the *aggregate* known-crop development set, select the strictest threshold that retains at least 90% of known crops, if possible. This is provisional; use holdout tests to challenge, not tune, the decision.

In [ ]:
TARGET_KNOWN_RETENTION=0.90
known=ydev!=OTHER
argmax=dev_prob.argmax(axis=1)
known_conf=dev_prob[:,:4].max(axis=1)
# Tie rejected by using > threshold, matching Android boundary convention.
# Choose a threshold among known-crop dev confidences, plus 0.0.
candidates=np.unique(np.r_[0.0,known_conf[known]])
feasible=[float(t) for t in candidates if np.mean((argmax[known]!=OTHER)&(known_conf[known]>t))>=TARGET_KNOWN_RETENTION]
threshold=max(feasible) if feasible else 0.0
if not feasible: print('WARNING: target known retention is not met even at threshold=0')
accepted=(argmax!=OTHER)&(known_conf>threshold)
print('Provisional known crop threshold:',threshold)
print('Development known-crop retention:',np.mean(accepted[known]))
print('Development seen-other crop rejection:',np.mean(~accepted[~known]))
by_class=[]
for i,name in enumerate(CLASSES):
    m=ydev==i
    by_class.append({'Class':name,'Count':int(m.sum()),'Classification accuracy':float(np.mean(argmax[m]==i)),
                     'Accepted for disease diagnosis':int(accepted[m].sum()),
                     'Accepted fraction':float(np.mean(accepted[m]))})
perclass=pd.DataFrame(by_class)
display(perclass)
perclass.to_csv(OUT/'dev_by_crop.csv',index=False)
score_table=dev_df[['RelativePath','Folder','Role','CropLabel','CropTarget','Group']].copy()
for i,name in enumerate(CLASSES):score_table['Prob_'+name.replace(' ','_')]=dev_prob[:,i]
score_table['Predicted']=np.asarray(CLASSES)[argmax]
score_table['Accepted']=accepted
score_table.to_csv(OUT/'stageB_dev_scores.csv',index=False)

## Cell 12 — Evaluate untouched held-out crop species **once**

Peach, raspberry, strawberry were fully removed before model training, checkpoint selection, and threshold calibration. **Do not retune on this result**, or it stops being an independent species holdout. These images are still PlantDoc-like; additional external camera datasets are required for trustworthy open-set performance.

In [ ]:
unseen_logits=BEST_MODEL.predict(unseen_ds.map(lambda x,y:x),verbose=0)
unseen_prob=softmax(unseen_logits.astype(np.float64),axis=1)
unseen_pred=unseen_prob.argmax(axis=1)
unseen_conf=unseen_prob[:,:4].max(axis=1)
unseen_accepted=(unseen_pred!=OTHER)&(unseen_conf>threshold)
unseen_scores=unseen[['RelativePath','Folder','Role','Group','CropLabel']].copy()
for i,name in enumerate(CLASSES):unseen_scores['Prob_'+name.replace(' ','_')]=unseen_prob[:,i]
unseen_scores['Predicted']=np.asarray(CLASSES)[unseen_pred]
unseen_scores['IncorrectlyAcceptedAsSupported']=unseen_accepted
unseen_scores.to_csv(OUT/'unseen_crop_scores.csv',index=False)
rows=[]
for folder,sub in unseen_scores.groupby('Folder'):
    rows.append({'Unseen species folder':folder,'Images':len(sub),
                 'Correctly rejected':int((~sub.IncorrectlyAcceptedAsSupported).sum()),
                 'Rejection fraction':float((~sub.IncorrectlyAcceptedAsSupported).mean()),
                 'False-supported fraction':float(sub.IncorrectlyAcceptedAsSupported.mean())})
novel=pd.DataFrame(rows)
display(novel)
novel.to_csv(OUT/'unseen_by_crop.csv',index=False)
print('Unseen species total:',len(unseen),'Rejection:',float(np.mean(~unseen_accepted)),
      'False supported acceptance:',float(np.mean(unseen_accepted)))

## Cell 13 — Save model, reproducibility metadata and reports

In [ ]:
FINAL=OUT/'crop_detector_stageB.keras'
shutil.copy2(BEST,FINAL)
metadata={
    'phase':'Phase9_StageB_Part1','task':'supported_crop_recognition',
    'class_order':CLASSES,'other_class_id':OTHER,'selected_checkpoint':chosen,
    'architecture':'MobileNetV2 alpha=0.35 input160x160 ImageNet pretrained',
    'input_shape':[1,160,160,3],'input_range':[0,255],
    'preprocessing':'EXIF orientation; RGB; PIL bilinear resize 160x160; internal [-1,1] normalization',
    'output':'five uncalibrated crop logits',
    'decision':'accept only if top class is not Other Crop AND top known-class softmax > threshold',
    'provisional_crop_acceptance_threshold':threshold,
    'target_dev_known_retention':TARGET_KNOWN_RETENTION,
    'train_images':len(train_df),'development_images':len(dev_df),
    'unseen_species_images':len(unseen),'unseen_species':UNSEEN_SPECIES,
    'source_manifest_sha256':hashlib.sha256(SOURCE_ZIP.read_bytes()).hexdigest(),
    'plantdoc_commit':COMMIT,
    'model_file':FINAL.name,'model_sha256':sha256_file(FINAL),
    'limitations':['Only one image-source family, PlantDoc, for all five categories',
       'Unseen crop species are a limited pre-registered stress test, not full field validation',
       'Stage B does not diagnose diseases, even when a supported species is recognized',
       'Development data is used for model and threshold selection',
       'Future evaluation needs distinct source domains and phone camera inputs']}
(OUT/'stageB_config.json').write_text(json.dumps(metadata,indent=2))
report={
  'selected_checkpoint':chosen,
  'development_accuracy':float(np.mean(dev_pred==ydev)),
  'development_balanced_accuracy':float(balanced_accuracy_score(ydev,dev_pred)),
  'known_dev_retention_at_gate':float(np.mean(accepted[known])),
  'seen_other_dev_rejection_at_gate':float(np.mean(~accepted[~known])),
  'unseen_other_rejection_at_gate':float(np.mean(~unseen_accepted)),
  'unseen_other_false_supported_acceptance':float(np.mean(unseen_accepted)),
  'threshold':threshold,'train_images':len(train_df),'dev_images':len(dev_df),
  'unseen_images':len(unseen)}
(OUT/'stageB_metrics.json').write_text(json.dumps(report,indent=2))
print('Saved:',FINAL,'size MiB:',round(FINAL.stat().st_size/1024**2,3))
print(json.dumps(report,indent=2))

## Cell 14 — Download results ZIP

Upload this ZIP in our conversation so we can inspect **per-class performance, threshold tradeoffs, and previously unseen crop rejections** before INT8 conversion and Android integration. Keep the earlier Stage A/C models as they are.

In [ ]:
archive=shutil.make_archive(str(ROOT/'CropDetector_Phase9_StageB_Part1_Results'),'zip',OUT)
print('Result archive:',archive)
with ZipFile(archive) as z:
    for item in z.infolist():print(item.filename,round(item.file_size/1024,1),'KiB')
files.download(archive)